# Budget LLM Fine-Tuning with QLoRA (Kaggle version)
### Phi-3-mini-4k-instruct on a free Kaggle T4 GPU

This notebook fine-tunes a small open-source LLM using **QLoRA** (4-bit quantization + LoRA adapters), so it fits and trains on a free Kaggle GPU (T4 x2 or P100).

**Before you start (Kaggle-specific):**
- In the right sidebar: **Settings > Accelerator > GPU T4 x2** (or P100).
- In the right sidebar: **Settings > Internet > On** (required to install packages and download the model/dataset).
- Kaggle free GPU quota is ~30 hrs/week — this notebook uses ~2 hrs for one full training run.

**Pipeline:**
1. Install dependencies
2. Load a dataset from Hugging Face
3. Load the base model in 4-bit (NF4) precision
4. Attach LoRA adapters to Phi-3's **actual** layer names (train ~1-2% of parameters)
5. Fine-tune with `SFTTrainer`
6. Evaluate: base model vs fine-tuned model, side by side
7. Save adapters to `/kaggle/working/` (persists as notebook output — no separate download step needed, though you can also download manually)

**Fixes included in this version** (from real debugging on a first run):
- `target_modules` corrected to Phi-3's actual fused layer names (`qkv_proj`, `gate_up_proj`, `down_proj`, `o_proj`) instead of Llama-style names (`q_proj`/`k_proj`/`v_proj`) — the original mistake silently trained only `o_proj`.
- `fp16=True` instead of `bf16=True` — T4/P100 don't support bf16.
- `gradient_checkpointing=False` — avoids a `trl`/`transformers` version incompatibility (`offload` kwarg error).
- `generate()` correctly slices only the newly generated tokens and uses `no_repeat_ngram_size=3` to avoid repetition loops.


## Step 1: Install dependencies

In [26]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl


## Step 2: Load and inspect the dataset

We use `b-mc2/sql-create-context` — a text-to-SQL dataset. It's easy to demo and easy to evaluate (exact-match accuracy), which makes for a clean before/after story on a resume.

Swap this out for any other Hugging Face dataset — a narrow, coherent dataset makes a stronger project than a generic one.


In [27]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(dataset)
print(dataset[0])

# Keep this small for a fast first run. Increase once the pipeline works end-to-end.
dataset = dataset.shuffle(seed=42).select(range(2000))

# Hold out a small eval set for the before/after comparison in Step 8
split_dataset = dataset.train_test_split(test_size=0.02, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")


Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
Train size: 1960, Eval size: 40


## Step 3: Format the data into a prompt template

Consistent formatting so the model learns a fixed input -> output pattern. This uses Phi-3's instruction format.


In [28]:
def format_example(example):
    prompt = (
        f"<|user|>\n"
        f"Given this SQL table schema: {example['context']}\n"
        f"Write a SQL query for: {example['question']}<|end|>\n"
        f"<|assistant|>\n{example['answer']}<|end|>"
    )
    return {"text": prompt}

train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

print(train_dataset[0]["text"])


<|user|>
Given this SQL table schema: CREATE TABLE table_name_31 (engine VARCHAR, chassis VARCHAR, driver VARCHAR)
Write a SQL query for: What engine was used when Aguri Suzuki drove the FA13B FA14 Chassis?<|end|>
<|assistant|>
SELECT engine FROM table_name_31 WHERE chassis = "fa13b fa14" AND driver = "aguri suzuki"<|end|>


## Step 4: Load the base model in 4-bit (NF4) precision

Weights are stored in 4-bit `NF4` (a data type optimized for normally-distributed weight values, so it loses less information than naive 4-bit rounding) instead of 16-bit, cutting memory use roughly 4x.

Note: no `trust_remote_code=True` — using transformers' own built-in Phi-3 implementation avoids a `rope_scaling` config incompatibility with custom remote code.


In [29]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # float16 (not bfloat16) — T4/P100 don't support bf16
    bnb_4bit_use_double_quant=True,          # also quantizes the quantization constants
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # Phi-3 has no pad token by default

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

## Step 5: Attach LoRA adapters

Instead of updating all ~3.8 billion parameters, we freeze the base model and train small low-rank adapter matrices injected into specific layers. `r=16` sets the rank (capacity) of these adapters.

**Important:** Phi-3 fuses its attention projections into a single `qkv_proj` layer and its MLP into `gate_up_proj`/`down_proj` — it does **not** have separate `q_proj`/`k_proj`/`v_proj` like Llama-style models. Targeting the wrong names means PEFT silently attaches adapters to only whichever name happens to match (usually just `o_proj`), training a much smaller slice of the model than intended. You can verify the real layer names yourself with:
```python
for name, _ in model.named_modules():
    if "proj" in name:
        print(name)
```


In [30]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],  # Phi-3's actual fused layer names
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# This line is your resume metric — expect a noticeably higher trainable% than
# you'd get from only targeting o_proj (a common mistake with Phi-3).


trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


## Step 6: Configure and run training

`fp16=True` (not `bf16`) since T4/P100 don't support bf16. `gradient_checkpointing=False` avoids a version-compatibility error between `trl` and `transformers` (an `offload` kwarg mismatch) — this uses a bit more GPU memory but fits fine for this dataset size at batch size 2.


In [31]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=False,
    loss_type="nll",
    completion_only_loss=True,   # <-- new: only train on the SQL answer, not the prompt
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.647739
20,0.971947
30,0.935076
40,0.907547
50,0.878965
60,0.915997
70,0.839620
80,0.867491
90,0.858070
100,0.780652


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=735, training_loss=0.6860360006085869, metrics={'train_runtime': 8491.706, 'train_samples_per_second': 0.692, 'train_steps_per_second': 0.087, 'total_flos': 1.273296284928e+16, 'train_loss': 0.6860360006085869, 'entropy': 0.5305107608437538, 'mean_token_accuracy': 0.8536270409822464, 'num_tokens': 503721.0, 'epoch': 3.0})

## Step 7: Save the adapter (and optionally merge)

Saving to `/kaggle/working/` means it persists as part of this notebook's output — you can download it from the notebook's "Output" tab, or re-attach it in a future session without retraining.


In [32]:
model.save_pretrained("/kaggle/working/phi3-qlora-adapter")
tokenizer.save_pretrained("/kaggle/working/phi3-qlora-adapter")

# Optional: zip it for a single-file download from the Output tab
!zip -rq /kaggle/working/adapter.zip /kaggle/working/phi3-qlora-adapter

# Optional: merge adapters into the base model for standalone deployment.
# Requires reloading the base model in fp16 (not 4-bit) to merge cleanly.
MERGE = False
if MERGE:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    )
    from peft import PeftModel
    merged_model = PeftModel.from_pretrained(base_model, "/kaggle/working/phi3-qlora-adapter")
    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained("/kaggle/working/phi3-merged")
    tokenizer.save_pretrained("/kaggle/working/phi3-merged")


## Step 8: Evaluate — base model vs fine-tuned model

This is the part that actually makes the project resume-worthy: a concrete before/after comparison, not just "I ran a training script."

`no_repeat_ngram_size=3` hard-blocks repeated 3-token sequences, which prevents the model from degenerating into loops (a common issue with small/undertrained models on greedy decoding). The function also correctly slices out only the newly generated tokens rather than decoding the full prompt+output sequence.


In [33]:
import re

def generate(model, tokenizer, question, context, max_new_tokens=100):
    model.config.use_cache = True  # re-enable cache for generation (was off for training)
    prompt = (
        f"<|user|>\nGiven this SQL table schema: {context}\n"
        f"Write a SQL query for: {question}<|end|>\n<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Only decode the newly generated tokens, not the prompt
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def normalize_sql(q):
    return re.sub(r"\s+", " ", q.strip().lower().rstrip(";"))

# Qualitative comparison on a few held-out examples
print("=" * 80)
for i in range(3):
    ex = eval_dataset[i]
    print(f"Question: {ex['question']}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Fine-tuned model output: {generate(model, tokenizer, ex['question'], ex['context'])}")
    print("=" * 80)

# Quantitative: exact-match accuracy over the eval set
correct = 0
for ex in eval_dataset:
    pred = generate(model, tokenizer, ex['question'], ex['context'])
    if normalize_sql(pred) == normalize_sql(ex['answer']):
        correct += 1
accuracy = correct / len(eval_dataset)
print(f"Exact-match accuracy on {len(eval_dataset)} held-out examples: {accuracy:.1%}")


Question: How many Goals against have Played more than 34?
Ground truth: SELECT COUNT(goals_against) FROM table_name_97 WHERE played > 34


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Fine-tuned model output: SELECT Given Given Given given Given Given Consider Given Given The Given Given Since Given Given If Given Given Which Given Given Based Given Given This Given Given In Given Given There Given Given To Given Given Assuming Given Given As Given Given What Given Given How Given Given You Given Given Your Given Given A Given Given For Given Given ### Given Given While Given Given When Given Given Write Given Given Without Given Given Suppose Given Given It Given Given
 Given Given0 Given Given G Given Given Input Given Given Give Given Given - Given Given
Question: What are the names listed in the family  il-1f2?
Ground truth: SELECT name FROM table_29871617_1 WHERE family_name = "IL-1F2"
Fine-tuned model output: SELECT Given Given Given given Given Given Consider Given Given The Given Given Since Given Given If Given Given Which Given Given Based Given Given This Given Given In Given Given There Given Given Assuming Given Given As Given Given To Given Given Write

In [34]:
model.save_pretrained("/kaggle/working/phi3-qlora-adapter")
tokenizer.save_pretrained("/kaggle/working/phi3-qlora-adapter")
!zip -rq /kaggle/working/adapter.zip /kaggle/working/phi3-qlora-adapter

In [25]:
print(type(model))
print(model.peft_config)
model.print_trainable_parameters()

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path='microsoft/Phi-3-mini-4k-instruct', revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'gate_up_proj', 'down_proj', 'qkv_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdl

## Notes for your resume / interview prep

- **Resume line:** *"Fine-tuned Phi-3-mini using QLoRA (4-bit NF4 quantization + LoRA adapters across attention and MLP layers), training only ~1-2% of parameters on a single T4 GPU, achieving [X]% exact-match accuracy on text-to-SQL generation."* Fill in your actual accuracy number from Step 8.
- **Be ready to explain:**
  - Why 4-bit NF4 loses less information than naive quantization (shaped for normally-distributed weights).
  - Why LoRA only needs to train a small fraction of parameters (injects low-rank update matrices rather than updating frozen base weights).
  - Why matching `target_modules` to the model's real architecture matters — Phi-3 fuses q/k/v and gate/up projections into single layers, unlike Llama's separate ones; targeting the wrong names silently trains only a fraction of what you intended.
- **Common issues on Kaggle:**
  - *GPU not detected / falling back to CPU*: check `Settings > Accelerator` is set to GPU, and confirm with `torch.cuda.is_available()`.
  - *No internet access*: check `Settings > Internet` is On, or `pip install` and `load_dataset` calls will fail.
  - *Session timeout*: Kaggle sessions can idle out; `save_strategy="epoch"` protects you from losing all training progress.
  - *Out of memory*: drop `per_device_train_batch_size` to 1 and raise `gradient_accumulation_steps` to 8 to keep the same effective batch size.
